# ZMUM Etap 1 — Uzupełnienie: Imputacja, Model bazowy, Krzywa uczenia

In [1]:
from sklearn.experimental import enable_iterative_imputer
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import KNNImputer, IterativeImputer
from feature_engine.imputation import MeanMedianImputer, CategoricalImputer, RandomSampleImputer

train = pd.read_csv("data/meetings_train.csv")
test  = pd.read_csv("data/meetings_test.csv")

# POPRAWKA: outcome==1 oznacza match (dec_woman==1 AND dec_man==1),
# outcome==4 oznacza brak zainteresowania z obu stron.
# Poprzednia wersja (outcome==4) przewidywala BRAK matchu jako target=1 -- blad.
train["match"] = (train["outcome"] == 1).astype(int)
test["match"]  = (test["outcome"]  == 1).astype(int)

# Usuń kolumny-identyfikatory: nie niosą informacji predykcyjnej (ryzyko data leakage)
ID_COLS = ["iid_woman", "wave_woman", "order_woman", "pid_woman",
           "iid_man",   "wave_man",   "order_man",   "pid_man"]
train = train.drop(columns=[c for c in ID_COLS if c in train.columns])
test  = test.drop(columns=[c for c in ID_COLS if c in test.columns])

print("Train:", train.shape, "| match rate:", train["match"].mean().round(3))
print("Test:",  test.shape,  "| match rate:", test["match"].mean().round(3))

# Baseline -- klasyfikator wiekszosciowy
majority_baseline = max(Counter(train["match"]).values()) / len(train["match"])
print(f"Majority baseline: {majority_baseline:.4f}")


Train: (3347, 166) | match rate: 0.165
Test: (837, 166) | match rate: 0.165
Majority baseline: 0.8351


/tmp/ipykernel_333843/1207445632.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["match"] = (train["outcome"] == 1).astype(int)
/tmp/ipykernel_333843/1207445632.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["match"]  = (test["outcome"]  == 1).astype(int)


## 1. Wykres 3 zmiennych: wiek vs atrakcyjność (kolor = match, rozmiar = fun)

In [2]:
numeric = ["age_woman", "attr1_1_woman", "fun1_1_woman", "match"]
scatter_df = train[numeric].dropna()

fig, ax = plt.subplots(figsize=(10, 7))
colors = scatter_df["match"].map({0: "#4575b4", 1: "#d73027"})
sizes  = (scatter_df["fun1_1_woman"] / scatter_df["fun1_1_woman"].max() * 150 + 20).clip(20, 170)

sc = ax.scatter(
    scatter_df["age_woman"],
    scatter_df["attr1_1_woman"],
    c=colors, s=sizes, alpha=0.5, linewidths=0.3, edgecolors="white"
)
ax.set_xlabel("Wiek kobiety", fontsize=12)
ax.set_ylabel("Preferowana atrakcyjność (attr1_1_woman)", fontsize=12)
ax.set_title("Wiek vs oczekiwana atrakcyjność\nKolor: match (czerwony) / no-match (niebieski)\nRozmiar: ocena humoru (fun)", fontsize=13)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor="#d73027", markersize=10, label="Match"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="#4575b4", markersize=10, label="No match"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="gray", markersize=6, label="fun = niski"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="gray", markersize=12, label="fun = wysoki"),
]
ax.legend(handles=legend_elements, loc="upper right")
plt.tight_layout()
plt.savefig("out/scatter_3var.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved. Punktów: {len(scatter_df)}")


Saved. Punktów: 3280


**Interpretacja wykresu scatter:** Punkty czerwone (match) i niebieskie (no match) silnie sie przeplataja -- wiek kobiety ani oczekiwana atrakcyjnosc nie separuja dobrze obu klas. Rozmiar punktu (fun) rowniez nie tworzy wyraznych skupien, co sugeruje, ze model potrzebuje bardziej zlozonych cech lub interakcji miedzy zmiennymi.

## 2. Imputacja — 5 metod

Brakujące wartości (48 447 z 3347 wierszy × 173 kolumn = 8,4%).

### Metody proste (feature-engine):
- **Mean** – zastąpienie średnią (MeanMedianImputer, strategy=mean)
- **Median** – zastąpienie medianą (MeanMedianImputer, strategy=median)
- **Most frequent** – dominanta/most frequent (MeanMedianImputer, strategy=most_frequent)

### Metody zaawansowane (sklearn):
- **KNNImputer** – k=5 najbliższych sąsiadów (relacyjne wzorce)
- **IterativeImputer** – wielokrotna regresja (MICE), max_iter=5


In [3]:
from sklearn.experimental import enable_iterative_imputer
numeric_cols = train.select_dtypes(include="number").columns.tolist()
# Wyklucz target, outcome oraz kolumny-identyfikatory
EXCLUDE = {"outcome", "match", "iid_woman", "wave_woman", "order_woman", "pid_woman",
           "iid_man", "wave_man", "order_man", "pid_man"}
numeric_cols = [c for c in numeric_cols if c not in EXCLUDE]

# Usuń kolumny z >90% brakujących wartości
numeric_cols = [c for c in numeric_cols if train[c].notna().sum() > len(train)*0.1]
X_train = train[numeric_cols]
X_test  = test[numeric_cols]
y_train = train["match"]
y_test  = test["match"]

print(f"Liczba cech: {len(numeric_cols)}")

results_imputation = {}

# Metoda 1: Mean (feature-engine)
imp_mean = MeanMedianImputer(imputation_method="mean", variables=numeric_cols)
X_tr_mean = imp_mean.fit_transform(X_train)
X_tr_mean = pd.DataFrame(X_tr_mean, columns=numeric_cols)
X_te_mean = imp_mean.transform(X_test)
X_te_mean = pd.DataFrame(X_te_mean, columns=numeric_cols)
results_imputation["Mean"] = (X_tr_mean, X_te_mean)
print("Mean: missing po imputacji:", X_tr_mean.isna().sum().sum())

# Metoda 2: Median (feature-engine)
imp_med = MeanMedianImputer(imputation_method="median", variables=numeric_cols)
X_tr_med = imp_med.fit_transform(X_train)
X_tr_med = pd.DataFrame(X_tr_med, columns=numeric_cols)
X_te_med = imp_med.transform(X_test)
X_te_med = pd.DataFrame(X_te_med, columns=numeric_cols)
results_imputation["Median"] = (X_tr_med, X_te_med)
print("Median: missing po imputacji:", X_tr_med.isna().sum().sum())

# Metoda 3: Most Frequent (sklearn)
from sklearn.impute import SimpleImputer
imp_freq = SimpleImputer(strategy="most_frequent")
X_tr_freq = pd.DataFrame(imp_freq.fit_transform(X_train), columns=numeric_cols)
X_te_freq = pd.DataFrame(imp_freq.transform(X_test), columns=numeric_cols)
results_imputation["Most_Frequent"] = (X_tr_freq, X_te_freq)
print("Most_Frequent: missing po imputacji:", X_tr_freq.isna().sum().sum())

# Metoda 4: KNNImputer (sklearn)
imp_knn = KNNImputer(n_neighbors=5)
X_tr_knn = pd.DataFrame(imp_knn.fit_transform(X_train), columns=numeric_cols)
X_te_knn = pd.DataFrame(imp_knn.transform(X_test), columns=numeric_cols)
results_imputation["KNN_k5"] = (X_tr_knn, X_te_knn)
print("KNN k=5: missing po imputacji:", X_tr_knn.isna().sum().sum())

# Metoda 5: HistGradientBoostingRegressor (zamiast MICE)
from sklearn.ensemble import HistGradientBoostingRegressor
imp_hgb = IterativeImputer(estimator=HistGradientBoostingRegressor(random_state=42), max_iter=5, random_state=42)
X_tr_hgb = pd.DataFrame(imp_hgb.fit_transform(X_train), columns=numeric_cols)
X_te_hgb = pd.DataFrame(imp_hgb.transform(X_test), columns=numeric_cols)
results_imputation["HGB"] = (X_tr_hgb, X_te_hgb)
print("HGB: missing po imputacji:", X_tr_hgb.isna().sum().sum())


Liczba cech: 154
Mean: missing po imputacji: 0
Median: missing po imputacji: 0
Most_Frequent: missing po imputacji: 0


KNN k=5: missing po imputacji: 0


/home/balicz3k/Documents/master-studies/ML/SpeedDatingExperiment/.venv/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


HGB: missing po imputacji: 0


## 3. Model bazowy — LogisticRegression na 10 seedach (boxplot)

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SEEDS = list(range(42, 52))
method_scores = {}

majority_baseline = max(Counter(y_train).values()) / len(y_train)

for method_name, (Xtr, Xte) in results_imputation.items():
    scores = []
    for seed in SEEDS:
        X_sub_tr, X_val, y_sub_tr, y_val = train_test_split(
            Xtr, y_train, test_size=0.2, random_state=seed, stratify=y_train
        )
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=500, random_state=seed, C=1.0))
        ])
        pipe.fit(X_sub_tr, y_sub_tr)
        scores.append(accuracy_score(y_val, pipe.predict(X_val)))
    method_scores[method_name] = scores
    mean_acc = np.mean(scores)
    improvement = mean_acc - majority_baseline
    print(f"{method_name:15s}: mean={mean_acc:.4f}  std={np.std(scores):.4f}  vs baseline={improvement:+.4f}")

print(f"\nMajority baseline: {majority_baseline:.4f}")
print("Uwaga: accuracy zblizona do baseline moze wskazywac na niezbalansowane klasy.")
print("Ponizej raport klasyfikacji dla imputacji Mean (seed=42):")

X_sub_tr, X_val, y_sub_tr, y_val = train_test_split(
    results_imputation["Mean"][0], y_train, test_size=0.2, random_state=42, stratify=y_train
)
pipe_report = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=42, C=1.0))
])
pipe_report.fit(X_sub_tr, y_sub_tr)
y_pred_report = pipe_report.predict(X_val)
accuracy = accuracy_score(y_val, y_pred_report)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Majority baseline: {majority_baseline:.4f}")
print(f"Improvement over baseline: {accuracy - majority_baseline:+.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_report, target_names=["No match", "Match"]))


Mean           : mean=0.8251  std=0.0091  vs baseline=-0.0100


Median         : mean=0.8272  std=0.0085  vs baseline=-0.0079


Most_Frequent  : mean=0.8267  std=0.0081  vs baseline=-0.0084


KNN_k5         : mean=0.8261  std=0.0066  vs baseline=-0.0090


HGB            : mean=0.8270  std=0.0080  vs baseline=-0.0081

Majority baseline: 0.8351
Uwaga: accuracy zblizona do baseline moze wskazywac na niezbalansowane klasy.
Ponizej raport klasyfikacji dla imputacji Mean (seed=42):

Accuracy: 0.8164
Majority baseline: 0.8351
Improvement over baseline: -0.0187

Classification Report:
              precision    recall  f1-score   support

    No match       0.83      0.97      0.90       560
       Match       0.07      0.01      0.02       110

    accuracy                           0.82       670
   macro avg       0.45      0.49      0.46       670
weighted avg       0.71      0.82      0.75       670



In [5]:
fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    [method_scores[m] for m in method_scores],
    patch_artist=True,
    boxprops=dict(facecolor="#90caf9", color="#1565c0"),
    medianprops=dict(color="#c62828", linewidth=2)
)
# Set labels manually
ax.set_xticklabels(list(method_scores.keys()))
ax.set_ylabel("Accuracy (walidacja)", fontsize=12)
ax.set_title("Model bazowy (Logistic Regression) — 10 seedów × 5 metod imputacji", fontsize=13)
ax.set_ylim(0.5, 0.75)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("out/boxplot_seeds.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved boxplot_seeds.png")

Saved boxplot_seeds.png


**Interpretacja boxplotu:** Wszystkie metody imputacji daja zblizona accuracy (~0.68-0.72), co oznacza, ze wybor metody imputacji ma niewielki wplyw na wynik klasyfikatora liniowego. Wartosci mediany sa bliskie baseline klasyfikatora wiekszosciowego (~0.675), dlatego nalezy stosowac precision/recall dla klasy Match jako glowna metryke jakosci.

## 4. Krzywa uczenia (Learning Curve)

In [6]:
# Używamy danych z imputacją Mean (najszybsza metoda)
X_lc, y_lc = results_imputation["Mean"][0], y_train

pipe_lc = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=42))
])

train_sizes, train_scores, val_scores = learning_curve(
    pipe_lc, X_lc, y_lc,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring="accuracy", n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", color="#1565c0", label="Trening")
ax.fill_between(train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.15, color="#1565c0")
ax.plot(train_sizes, val_scores.mean(axis=1), "o-", color="#c62828", label="Walidacja (CV)")
ax.fill_between(train_sizes,
    val_scores.mean(axis=1) - val_scores.std(axis=1),
    val_scores.mean(axis=1) + val_scores.std(axis=1),
    alpha=0.15, color="#c62828")
ax.set_xlabel("Rozmiar zbioru treningowego", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Krzywa uczenia — Logistic Regression (imputacja: Mean)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("out/learning_curve.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved learning_curve.png")
print(f"Walidacja (max): {val_scores.mean(axis=1).max():.4f}")


Saved learning_curve.png
Walidacja (max): 0.8276


**Interpretacja krzywej uczenia:** Rozbieznosc miedzy accuracy treningowa a walidacyjna maleje wraz ze wzrostem danych -- model nie jest silnie przeuczony. Plateau walidacji wskazuje, ze samo zwiekszenie zbioru treningowego nie poprawi istotnie wynikow; potrzebna jest bardziej zlozoona architektura modelu lub inzynieria cech.